<a href="https://colab.research.google.com/github/Haross/sql_notebook/blob/main/Midterm_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BTM-3300 Midterm Practice**

You will receive several independent questions using the following tables:


## The `cat` Table

All exercises use the same table:

Each row represents **one cat**, and each column stores an attribute of that cat.

| Column name | Description |
|------------|-------------|
| `id` | Unique identifier for each cat |
| `name` | Name of the cat |
| `breed` | Breed of the cat (e.g. Siamese, Persian, Ragdoll) |
| `coloration` | Fur color or pattern (may be `NULL`) |
| `age` | Age of the cat in years |
| `sex` | Sex of the cat: `'M'` (male) or `'F'` (female) |
| `fav_toy` | Favorite toy of the cat (may be `NULL`) |

## The `vet_visit` Table

Some exercises will also use a second table:

Each row represents **one vet visit**, linked to a cat via `cat_id`.

| Column name  | Description                                      |
| ------------ | ------------------------------------------------ |
| `visit_id`   | Unique identifier for each visit                 |
| `cat_id`     | The cat who had the visit (matches `cat.id`)     |
| `visit_date` | Date of the vet visit                            |
| `reason`     | Reason for the visit (e.g. Checkup, Vaccination) |
| `cost`       | Cost of the visit (in euros)                     |

**Take your time to **read each prompt carefully**, translate it into logic, and then into SQL.**


In [1]:
# @title
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

status_out = widgets.Output()

def update_status(title, message, type="info"):
    colors = {
        "info":    ("#1a73e8", "#eef3fe", "ℹ️"),
        "success": ("#2e7d32", "#eaf7ee", "✅"),
        "warning": ("#f9a825", "#fff8e1", "⚠️"),
        "error":   ("#c62828", "#fdecea", "❌"),
    }

    border, background, icon = colors.get(type, colors["info"])

    with status_out:
        status_out.clear_output()
        display(HTML(f"""
        <div style="
            padding:16px;
            border-left:6px solid {border};
            background:{background};
            border-radius:10px;
            font-family:Arial, sans-serif;
            line-height:1.6;
            box-shadow:0 2px 6px rgba(0,0,0,0.05);
        ">
            <div style="font-size:16px;font-weight:600;margin-bottom:6px;">
                {icon} {title}
            </div>
            <div style="font-size:14px;">
                {message}
            </div>
        </div>
        """))

display(status_out)


t = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""
if t:
    update_status(
        "Important: Before Starting",
        """
        The next cell will ask you to enter your <b>student token</b>.<br><br>
        After typing it, press <b>Enter</b>.<br>
        You only need to do this once.
        """,
        type="info"
    )
else:
    update_status(
      "Important: Before Starting",
      """
      The next cell will ask you to enter your <b>student token</b>.<br><br>
      After typing it, press <b>Enter</b>.<br>
      You only need to do this once.
      """,
      type="info"
    )


Output()

In [ ]:
# @title
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

existing = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""
if not existing:
    while True:
        token = input("Student token: ").strip()
        if token:
            break
        print("⚠️ Token cannot be empty. Try again.")

    TOKEN_FILE.write_text(token)
    update_status(
        "Token Saved Successfully",
        f"Your token has been stored securely.<br><br>Your token is:<b>{token}</b>",
        type="success"
    )
    print("✅ Token saved.")
else:
    token = existing
    update_status(
        "Token Already Saved",
        f"Your token has been already stored securely.<br><br>Your token is: <b>{token}</b>",
        type="success"
    )
    print("✅ Token already saved.")

In [3]:
# @title ER Diagram
%%html
<img id="er-img" style="width:20%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/practice_mt.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/practice_mt_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

## **SQL Environment Setup (do not edit).**

In [4]:
# @title
%%capture
!mkdir -p notebook_lib
!wget -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/7baff2c6485cdf641cabcdb55d92a51317cd18b9/notebook_lib/validators.py
!wget -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import sqlite3
import pandas as pd
from pathlib import Path


In [5]:
# @title

DB_FILE = 'kittens.db'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = sqlite3.connect(DB_FILE)
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript(r'''
DROP TABLE IF EXISTS cat;
-- Create table
CREATE TABLE cat (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    breed TEXT NOT NULL,
    coloration TEXT,
    age INTEGER,
    sex TEXT CHECK (sex IN ('M','F')),
    fav_toy TEXT
);

-- Insert rows
INSERT INTO cat (id, name, breed, coloration, age, sex, fav_toy) VALUES
(1, 'Micky',  'Maine Coon', 'tortoiseshell', 3,  'M', ' ball - red'),
(2, 'Nine',   'Ragamuffin', 'cream',         2,  'M', 'ball - green'),
(3, 'Carmen', 'Persian',    'brown',         2,  'F', 'toy mouse'),
(4, 'Luna',   'Abyssinian', 'tortoiseshell', 12, 'F', 'teaser'),
(5, 'Bella',  'Siamese',    'red',           15, 'F', 'teaser'),
(6, 'Tiger',  'Maine Coon', 'black',         12, 'M', 'toy mouse'),
(7, 'Shadow', 'Ragdoll',    'white',         16, 'M', NULL),
(8, 'Oreo',   'Siamese',    'red',           4,  'F', 'toy mouse'),
(9, 'Gizmo',  'Ragdoll',    'blue',          3,  'M', 'laser'),
(10,'Lucy',   'Persian',    'blue',          7,  'F', 'teaser'),


(11, 'Chloe',  'Maine Coon', 'brown', 4,  'F', 'toy mouse'),
(12, 'Misty',  'Maine Coon', 'gray',  6,  'F', 'ball - blue'),
(13, 'Penny',  'Siamese',    'cream', 8,  'F', 'laser'),
(14, 'Casey',  'Persian',    'white', 5,  'M', 'toy mouse');


DROP TABLE IF EXISTS vet_visit;

CREATE TABLE vet_visit (
    visit_id INTEGER PRIMARY KEY,
    cat_id INTEGER NOT NULL,
    visit_date TEXT NOT NULL,
    reason TEXT NOT NULL,
    cost INTEGER NOT NULL,
    FOREIGN KEY (cat_id) REFERENCES cat(id)
);

INSERT INTO vet_visit (visit_id, cat_id, visit_date, reason, cost) VALUES
(1, 1,  '2026-01-05', 'Vaccination', 50),
(2, 1,  '2026-02-10', 'Checkup', 40),
(3, 3,  '2026-01-20', 'Injury', 120),
(4, 4,  '2026-03-01', 'Dental', 200),
(5, 6,  '2026-01-15', 'Vaccination', 50),
(6, 8,  '2026-02-12', 'Checkup', 40),
(7, 11, '2026-01-25', 'Surgery', 300),
(8, 12, '2026-02-01', 'Vaccination', 50),
(9, 14, '2026-01-18', 'Checkup', 40),


(10, 8, '2026-01-05', 'Vaccination', 30),
(11, 8, '2026-01-28', 'Dental', 70),


(12, 11, '2026-01-10', 'Checkup', 100),
(13, 11, '2026-01-31', 'Injury', 50),

(14, 10, '2026-01-12', 'Checkup', 25),

(15, 13, '2026-01-01', 'Vaccination', 60),
(16, 13, '2026-01-15', 'Checkup', 40);
''')
print(f"Database ready ✅ ({DB_FILE})")


Database ready ✅ (kittens.db)


# Questions

In [6]:
# @title Q1 10 points

submitter = make_cloud_run_submitter(
    submit_url="https://cinemax-validator-830800716261.europe-west1.run.app/submit",
    exam_id="PRACTICE_1_2026",
    question_id="q1",
    api_key="SECRET123",
)


make_sql_runner(
    conn,
    runner_id="q1",
    description_md='### Practice 4.2\nReturn cats that:\n\n* have a recorded coloration          \n* have a recorded favorite toy          \n* are NOT (breed = Ragdoll OR age > 15)          \n* are either female or have a name ending with y           \n* and their favorite toy is not teaser\n',
    sol_sql="SELECT name, breed, coloration, age, sex, fav_toy FROM cat WHERE coloration IS NOT NULL AND fav_toy IS NOT NULL AND NOT (breed = 'Ragdoll' OR age > 15) AND (sex = 'F' OR name LIKE '%y') AND fav_toy <> 'teaser';",
    select_only=True,
    dedupe=True,
    schema_tables=['cat'],
     submitter=submitter,
)


In [7]:
# @title Q2 20 points
submitter = make_cloud_run_submitter(
    submit_url="https://cinemax-validator-830800716261.europe-west1.run.app/submit",
    exam_id="PRACTICE_1_2026",
    question_id="q2",
    api_key="SECRET123",
)



make_sql_runner(
    conn,
    runner_id="q2",
    description_md="""# 🐾 **Practice 2 – Vet Visit Summary**

Write a query that returns:

* the cat’s **name**
* the cat’s **breed**
* january_visits: the **number of vet visits in January 2026**
* total_spent: the **total amount spent in January 2026**

### Conditions:

1. Only include cats that:

   * are **female**
   * and are **younger than 10 years old**

2. Only count vet visits that:

   * occurred between `'2026-01-01'` and `'2026-01-31'` (inclusive)

3. Only include cats who had **at least 2 vet visits in January 2026**

4. Sort the results:

   * first by **total amount spent (descending)**
   * then by **cat name (ascending)**
    """,
    sol_sql="""
SELECT
    c.name,
    c.breed,
    COUNT(*) AS january_visits,
    SUM(v.cost) AS total_spent
FROM cat c
JOIN vet_visit v
    ON v.cat_id = c.id
WHERE c.sex = 'F'
  AND c.age < 10
  AND v.visit_date BETWEEN '2026-01-01' AND '2026-01-31'
GROUP BY c.id, c.name, c.breed
HAVING COUNT(*) >= 2
ORDER BY total_spent DESC, c.name ASC;
    """,
    select_only=True,
    dedupe=True,
    schema_tables=['cat'],
    submitter=submitter,
)


In [8]:
# @title Q3 100 points
submitter = make_cloud_run_submitter(
    submit_url="https://cinemax-validator-830800716261.europe-west1.run.app/submit",
    exam_id="PRACTICE_1_2026",
    question_id="q3",
    api_key="SECRET123",
)



make_sql_runner(
    conn,
    runner_id="q3",
    description_md="""Prepare a report with one row per breed showing:

* breed
* avg_age: the average age of cats in that breed
* unique_toys: the number of different favorite toys recorded for that breed

Apply the following rules:

Only consider cats that:

* have a recorded coloration
* have a recorded favorite toy

Exclude cats that meet **either** of the following risk conditions:

* the cat is 12 years old or older and its favorite toy contains the word “ball”
* the cat’s name begins with the letter “L” and the cat is female

In addition, only include cats whose breed:

* contains a space in its name, or
* ends with the letters “ese”

Only include breeds that:

* have at least two qualifying cats
* and whose average age is between 3 and 12 years (inclusive)

Order the result:

* by average age, from lowest to highest
* then by breed name alphabetically
    """,
    sol_sql="""
    SELECT
  breed,
  AVG(age) AS avg_age,
  COUNT(DISTINCT fav_toy) AS unique_toys
FROM cat
WHERE coloration IS NOT NULL
  AND fav_toy IS NOT NULL
  AND NOT (
        (age >= 12 AND fav_toy LIKE '%ball%')
     OR (name LIKE 'L%' AND sex = 'F')
  )
  AND (breed LIKE '% %' OR breed LIKE '%ese')
GROUP BY breed
HAVING COUNT(*) >= 2
   AND AVG(age) BETWEEN 3 AND 12
ORDER BY avg_age ASC, breed ASC;
    """,
    select_only=True,
    dedupe=True,
    schema_tables=['cat'],
    submitter=submitter,
)
